#### *Importo los path de los datasets a utilizar*


In [2]:
import csv
import sys
from pathlib import Path
sys.path.append(str(Path('.').resolve().parent.parent))
from modules.paths import AR_AIRPORTS_DATA_MODIFIED,AR_LAKES_MODIFIED,C2022_DATA_MODIFIED,CONNECTIONS_DATA_MODIFIED,AR_DATA

##### 2) Los aeropuertos de una elevación, (bajo, medio, alto) cuya especificación se permita  modificar fácilmente, evaluando la columna creada 'elevation_name'.

In [3]:
options=['bajo','medio','alto']
while True:
    elevation=input("ingrese la elevacion de los aeropuertos que desea encontrar (bajo,medio,alto)").strip().lower()
    if(elevation in options):
        break
    else:
        print('por favor, ingrese una opción correcta')
with AR_AIRPORTS_DATA_MODIFIED.open(mode='r',encoding='utf-8')as airport_csv:
    reader=csv.reader(airport_csv)
    next(reader)#para saltearme el encabezado
    list_airports=[]# aca voy a ir almacenando los nombre de los aeropuertos
    for row in reader:
        if row[23]==elevation: 
            list_airports.append(row[3])

print(list_airports)#imprime los aeropuertos que coinciden con la elevacion ingresada



['Minister Pistarini International Airport', 'Jorge Newbery Airpark', 'Malvinas Argentinas Airport', 'San Fernando Airport', 'Piloto Civil N. Fernández Airport', 'Rosario Islas Malvinas International Airport', 'Sauce Viejo Airport', 'Gobernador Ramón Trejo Noel International Airport', 'Ástor Piazzola International Airport', 'El Palomar Airport', 'Morón Airport', 'Don Torcuato Airport', 'Zarate Airport', 'Santa Teresita Airport', 'Gobernador Castello Airport', 'Comodoro Pierrestegui Airport', 'Ezpeleta Airport', 'Ushuaia Aeroclub Airport', 'Villa Gesell Airport', 'Escobar Aeroclub', 'Comandante Luis Piedrabuena Airport', 'General Rodriguez Airport', 'Mariano Moreno Airport', 'Isla Martin Garcia Airport', 'Matanza Airport', 'Lobos Airport', 'Luján Airport', 'Quilmes Airport', 'Trelew Aeroclub Airport', 'Goya Airport', 'Gualeguaychu Airport', 'La Plata Airport', 'Antoine de Saint Exupéry Airport', 'Baradero Airport', 'Benavidez Airport', 'Brandsen Airport', 'Campo de Mayo Military Airport

##### 3) Los aeropuertos que tienen una mayor o menor elevación con respecto al valor numérico dado.

##### Funciones que comparan la altura de los Aeropuertos con la ingresadas por el usuario, según el criterio elegido por el mismo

In [3]:
def is_greater(num1,num2):
    if num1 != '': #si el campo no esta vacio lo evaluo, sino devuelvo false
        return int(num1)>num2
    else:
        return False
def is_smaller(num1,num2):
    if num1 != '':
        return int(num1)<num2
    else:
        return False
    

In [9]:
while True:
  elevation_input=input('ingrese una elevacion en numeros, para listar los Aeropuertos superiores o menores a la misma:')
  if elevation_input.isdigit():
     elevation_number=int(elevation_input)
     break
  else:
     print('por favor ingrese un valor válido')
     
while True:
  options=['mayor', 'menor']    
  search_criteria=input("ingrese 'mayor' para listar los aeropuertos con mayor elevacion a la ingresada," 
                      " o ingrese 'menor' para lo contrario")
  if search_criteria in options:
     break
  else:
     print('por favor criterio de búsqueda válido')


list_airports=[]# Lista donde guardo los aeropuertos que cumplan
airport=3 # Columna donde se encuentra el nombre del aeropuerto
elevation=6 # Columna que indica la elevación del aeropuerto
try:
    with AR_AIRPORTS_DATA_MODIFIED.open(mode='r',encoding='utf-8')as airport_csv:
        reader=csv.reader(airport_csv)
        next(reader)# Salteo el encabezado
        # Filtro los Aeropuertos según el criterio ingresado
        if search_criteria == 'mayor':
          list_airports=[row[airport] for row in reader if is_greater(row[elevation],elevation_number)]
        else:
            list_airports=[row[airport] for row in reader if is_smaller(row[elevation],elevation_number)]
                                                                                      
except FileNotFoundError:
    print('Error, el archivo AR_AIRPORTS_DATA_MODIFIED no fue encontrado')
print(list_airports)

       

por favor ingrese un valor válido
por favor criterio de búsqueda válido
['Malvinas Argentinas Airport', 'El Calafate - Commander Armando Tola International Airport', 'Cataratas Del Iguazú International Airport', 'El Plumerillo Airport', 'San Carlos De Bariloche Airport', 'Martin Miguel De Guemes International Airport', 'Ingeniero Ambrosio Taravella Airport', 'Colonia Catriel Airport', 'General E. Mosconi Airport', 'Almirante Marco Andres Zar Airport', 'Capitan D Daniel Vazquez Airport', 'Presidente Peron Airport', 'Comandante Espora Airport', 'Libertador Gral D Jose De San Martin Airport', 'Santa Rosa Airport', 'Pergamino Airport', 'El Tehuelche Airport', 'Morón Airport', 'Coronel Felipe Varela International Airport', 'Aviador C. Campos Airport', 'Brigadier Antonio Parodi Airport', 'Puerto Deseado Airport', 'Santa Cruz Airport', 'Resistencia International Airport', 'Teniente Benjamin Matienzo Airport', 'Gobernador Horacio Guzman International Airport', 'Comodoro Pierrestegui Airport', 

#### 4) Los aeropuertos, lagos y tipo de conectividad en provincias con población mayor o menor a un valor que se pueda especificar fácilmente.

##### Defino una función para que recibe una palabra(una provincia) y retorna la misma sin las tildes

In [63]:
def remove_accents(word):
   
    accents = {'á': 'a', 'é': 'e', 'í': 'i', 'ó': 'o', 'ú': 'u', 'Á': 'A', 'É': 'E', 'Í': 'I', 'Ó': 'O', 'Ú': 'U'}
    

    for accented_char, non_accented_char in accents.items():
        word = word.replace(accented_char, non_accented_char)
    
    return word

##### Función que busca los lagos de una determinada provincia y retorna una lista con los mismos

In [64]:
def search_lakes(AR_LAKES_MODIFIED,prov):
    with AR_LAKES_MODIFIED.open(mode='r',encoding='utf-8')as lakes_csv:
      list_lakes=[]
      reader_lakes = csv.reader(lakes_csv)
      next(reader_lakes)
      for row_lakes in reader_lakes: #recorro toda las fila de lagos buscando la prov actual
            list_modified=[]
            date_one=[]
            date=row_lakes[1].split('/')
            if(len(date)>1):
                list_modified = [word.lower().strip() for word in date] #paso a minusucla y saco los espacios
                list_modified = [remove_accents(word) for word in list_modified]# saco los acentos
            else:
                 date_one=remove_accents(date[0].lower())
            if(prov in date_one or prov in list_modified):
                list_lakes.append(row_lakes[0])
    return list_lakes

##### Función que busca los Aerpouertos de una determinada provincia y retorna una lista con los mismos

In [65]:
def search_airports(AR_AIRPORTS_DATA_MODIFIED,prov):
      with  AR_AIRPORTS_DATA_MODIFIED.open(mode='r',encoding='utf-8')as airport_csv:
                   
          reader_airports=csv.reader(airport_csv)
          next(reader_airports)
          list_airports=[]
          for row_airports in reader_airports:
              prov_current=remove_accents(row_airports[24].lower()) #elimino minusucla y acentos
              if(prov_current==prov):
                 list_airports.append(row_airports[3])
      return list_airports

##### Función que busca los datos de la conectividad en una provincia y retorna una lista con un diccionario donde las claves son el tipo de conectividad y el valor es 'SI' o 'NO'  dependiendo si tiene la conectividad

In [102]:
def search_connection(CONNECTIONS_DATA_MODIFIED,prov):
            with CONNECTIONS_DATA_MODIFIED.open(mode='r',encoding='utf-8')as connection_csv:
                reader_connection=csv.reader(connection_csv)
                header=next(reader_connection)
                list_connection=[{'ADSL': 'NO','CABLEMODEM':'NO','DIALUP':'NO','FIBRAOPTICA':'NO','SATELITAL':
                                      'NO','WIRELESS':'NO','TELEFONIAFIJA':'NO','3G':'NO','4G':'NO'}]
                for row_connection in reader_connection:
                    prov_current=remove_accents(row_connection[0].lower())
                    if(prov_current in prov):
                        for i in range(4,13):
                          if(row_connection[i]=='SI'):
                             clave=header[i]
                             list_connection[0][clave]='SI'
            return list_connection

##### El usuario ingresa el numero de población y el criterio de búsqueda

In [103]:
population=int(input('ingrese un numero de población'))
options=['mayor','menor']
while True:
    bigger_smallest=input('ingrese mayor para buscar los datos en provincias con más población que el num '
                      'ingresado, caso contrario ingrese menor')
    if bigger_smallest in options:
        break
    else:
        print('ingrese una opción válida')

list_dicc=[] #voy a almacenar en una lista de diccionarios

##### Abro el archivo de censo y trabajo con los datos del mismo recorriendo cada fila y a su vez voy consultando los archivos de Aeropuertos, Conectividad y Lagos, para que generar el resultado pedido

In [104]:
with(
    C2022_DATA_MODIFIED.open(mode='r',encoding='utf-8') as censo_csv, #abro el archivo censos como lectura
    
):
    reader_censo=csv.reader(censo_csv)
    next(reader_censo)
    next(reader_censo)
    next(reader_censo)#para saltearme las 3 primeras filas que no tienen provincias

    for row in reader_censo: 
        prov_name=row[0]# me quedo con el nombre sin pasarlo a min,ni sacarle las tildes

        prov=row[0].lower() #me quedo con el nombre de la prov de la fila actual y paso a min

        prov= remove_accents(prov)#elimino las tildes

        prov_for_search_connection=prov.split(',') #correcion para un caso buscando la conexion de las provincias
        
        if (bigger_smallest=='mayor'):# pregunto que criterio de busqueda ingreso
            if(int(row[1])>population): #si la cant es mayor a la q ingreso
  
                list_lakes=search_lakes(AR_LAKES_MODIFIED,prov)

                list_airports=search_airports(AR_AIRPORTS_DATA_MODIFIED,prov)

                
                list_connection=search_connection(CONNECTIONS_DATA_MODIFIED,prov_for_search_connection)

                list_dicc.append({'Provincia:': prov_name, 'Lagos:':list_lakes, 'Aeropuertos': list_airports, 
                          'Conectividad':list_connection})  #agrego la prov ya procesada con todos los lagos
        else:
            if(int(row[1])<population): #si la cant es mayor a la q ingreso
  
                list_lakes=search_lakes(AR_LAKES_MODIFIED,prov)

                list_airports=search_airports(AR_AIRPORTS_DATA_MODIFIED,prov)

                list_connection=search_connection(CONNECTIONS_DATA_MODIFIED,prov_for_search_connection)
   
                list_dicc.append({'Provincia:': prov_name, 'Lagos:':list_lakes, 'Aeropuertos': list_airports, 
                          'Conectividad':list_connection})  #agrego la prov ya procesada con todos los lagos



#### Imprimo la lista de diccionarios generada: 

In [105]:
print(list_dicc)

[{'Provincia:': 'Buenos Aires', 'Lagos:': ['Lago Epecuén'], 'Aeropuertos': ['San Fernando Airport', 'Ástor Piazzola International Airport', 'El Palomar Airport', 'Pergamino Airport', 'Morón Airport', 'Don Torcuato Airport', 'Ezpeleta Airport', 'Héroes De Malvinas Airport', 'Villa Gesell Airport', 'San Andrés de Giles Airport', 'Escobar Aeroclub', 'Lobos Airport', 'Mercedes Airport', 'Quilmes Airport', 'Salto Airport', 'La Plata Airport', 'Comodoro Pedro Zanni Airport', 'Arrecifes Aeroclub Airport', 'Azul Airport', 'Baradero Airport', 'Bragado Airport', 'Brandsen Airport', 'Cañuelas Airport', 'Carmen de Areco Airport', 'Chacabuco Airport', 'Chascomús Airport', 'Chivilcoy Airport', 'Colón Airport', 'Dolores Airport', 'El Pajaro Airport', 'General Belgrano Airport', 'General Las Heras Aeroclub', 'Los Toldos Airport', 'Tolosa Airport', 'Las Flores Airport', 'Lincoln Airport', 'La Caida Airport', 'Marcos Paz Airfield', 'Valle Del Conlara International Airport', 'Navarro Airport', 'La Noria 

##### 5) Mostrar los aeropuertos en las capitales de cada provincia.

In [122]:
with AR_DATA.open(mode='r', encoding='utf-8') as ar_cvs:  
    ar_reader = csv.DictReader(ar_cvs)

    list_result = []

    # Abre el archivo AR_AIRPORTS_DATA_MODIFIED dentro del bloque with
    with AR_AIRPORTS_DATA_MODIFIED.open(mode='r', encoding='utf-8') as airport_csv:
        airports_reader = csv.DictReader(airport_csv)

        for row in airports_reader:
            city = row['municipality'] #guardo la municip de los aeropuertos
            airport = row['name']# guardo el nombre del Aeropuerto actual

            # Reinicia el cursor de lectura del archivo ar_cvs al principio antes de iterar sobre él
            ar_cvs.seek(0)

            for row_ar in ar_reader:
                if row_ar['city'] == city:
                    if row_ar['capital'] == 'admin':
                        list_result.append({'Capital': city, 'Aeropuerto': airport})

    print(list_result)

[{'Capital': 'Ushuaia', 'Aeropuerto': 'Malvinas Argentinas Airport'}, {'Capital': 'Mendoza', 'Aeropuerto': 'El Plumerillo Airport'}, {'Capital': 'Salta', 'Aeropuerto': 'Martin Miguel De Guemes International Airport'}, {'Capital': 'Rawson', 'Aeropuerto': 'Almirante Marco Andres Zar Airport'}, {'Capital': 'Santa Fe', 'Aeropuerto': 'Sauce Viejo Airport'}, {'Capital': 'Posadas', 'Aeropuerto': 'Libertador Gral D Jose De San Martin Airport'}, {'Capital': 'Santa Rosa', 'Aeropuerto': 'Santa Rosa Airport'}, {'Capital': 'Catamarca', 'Aeropuerto': 'Coronel Felipe Varela International Airport'}, {'Capital': 'Resistencia', 'Aeropuerto': 'Resistencia International Airport'}, {'Capital': 'San Miguel de Tucumán', 'Aeropuerto': 'Teniente Benjamin Matienzo Airport'}, {'Capital': 'San Salvador de Jujuy', 'Aeropuerto': 'Gobernador Horacio Guzman International Airport'}, {'Capital': 'Ushuaia', 'Aeropuerto': 'Ushuaia Aeroclub Airport'}, {'Capital': 'La Rioja', 'Aeropuerto': 'Capitan V A Almonacid Airport'},